# LeetCode 4104 — Count Valid Substrings II (Medium)

Bài toán LeetCode:

```python
class Solution:
    def countValidSubstrings(self, s: str) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4104 (Biweekly Contest 195 — câu 3).

Bài này là "người anh" của bài 4102. Đề gần như y hệt, chỉ khác một chữ: cấm `"abc"` như một **subsequence (dãy con)** chứ không phải substring. Một chữ đó làm thay đổi hoàn toàn cách giải.

Kỹ thuật chính:

- **Khớp dãy con bằng tham lam** (greedy subsequence matching).
- **Bảng "vị trí xuất hiện kế tiếp"** (next-occurrence table).
- **Chuyền vị trí qua từng chặng** — một dạng quy hoạch động chỉ dùng `O(1)` bộ nhớ.

---

## Sau notebook này, bạn sẽ nắm được

1. Cách kiểm tra `t` có phải subsequence của một chuỗi không, và vì sao "khớp sớm nhất" là tối ưu.
2. Brute Force `O(n³)` và cải tiến `O(n²)` bằng "máy trạng thái" đếm số ký tự đã khớp.
3. Lời giải `O(n)` theo **điểm đầu**: dùng bảng vị trí kế tiếp.
4. Lời giải `O(n)` theo **điểm cuối**, `O(1)` bộ nhớ: chuyền vị trí `a → ab → abc`.
5. Chứng minh ngắn gọn, phân tích độ phức tạp, kiểm thử chéo.
6. Lỗi hay gặp, mẹo nhận diện và các bài tập mở rộng (kể cả mẫu `t` tổng quát).

# 1. Đề bài yêu cầu gì?

Cho chuỗi `s` chỉ gồm `'a'`, `'b'`, `'c'`.

Một chuỗi được gọi là **hợp lệ** nếu nó **không chứa `"abc"` như một subsequence** — tức là **không** tồn tại ba vị trí `p < q < r` mà `a` ở `p`, `b` ở `q`, `c` ở `r` (không cần liền nhau).

Đếm số **substring** (chuỗi con liên tiếp) hợp lệ của `s`. Các substring khác vị trí được tính riêng.

> Để ý: ta vẫn **đếm substring**, nhưng điều kiện "hợp lệ" của mỗi substring được kiểm tra theo **subsequence**.

---

## Ví dụ 1

```python
s = "abac"   ->   9
```

Có 10 substring. Chỉ có `"abac"` (cả chuỗi) chứa `a, b, c` theo thứ tự (`a` ở 0, `b` ở 1, `c` ở 3) → không hợp lệ. Còn 9:

```text
"a", "ab", "aba", "b", "ba", "bac", "a", "ac", "c"
```

So với bài 4102: ở đó `"abac"` hợp lệ (vì không có `"abc"` **liền nhau**), đáp án là 10.

## Ví dụ 2

```python
s = "abc"   ->   5
```

`"a", "ab", "b", "bc", "c"` hợp lệ; `"abc"` thì không.

## Ví dụ 3

```python
s = "ac"   ->   3
```

Không có `b` nên mọi substring đều hợp lệ.

---

## Ràng buộc

```text
1 <= s.length <= 10^5
s chỉ gồm 'a', 'b', 'c'
```

# 2. Kiến thức nền tảng

## 2.1. Kiểm tra subsequence bằng tham lam

Muốn biết `t = "abc"` có là subsequence của `u` không, ta đi từ trái sang phải trên `u` và **khớp ký tự cần tìm sớm nhất có thể**:

```text
u = "x a y b b z c"
         ^   ^     ^
         a   b     c     -> khớp đủ 3 ký tự -> có
```

Vì sao khớp **sớm nhất** là đúng? Nếu có một cách khớp nào đó, thì việc chọn vị trí sớm hơn cho ký tự hiện tại chỉ để lại **nhiều chỗ hơn** cho các ký tự phía sau — không bao giờ thiệt.

In [1]:
def is_subsequence(t: str, u: str) -> bool:
    k = 0                          # đã khớp được k ký tự đầu của t
    for ch in u:
        if k < len(t) and ch == t[k]:
            k += 1
    return k == len(t)


print(is_subsequence("abc", "xaybbzc"))   # True
print(is_subsequence("abc", "abac"))      # True: a(0) b(1) c(3)
print(is_subsequence("abc", "acb"))       # False: c đứng trước b
print(is_subsequence("abc", "bca"))       # False

True
True
False
False


## 2.2. "Máy trạng thái": đã khớp được bao nhiêu ký tự?

Biến `k` trong hàm trên chính là **trạng thái**: `k = 0, 1, 2, 3` nghĩa là đã khớp được `""`, `"a"`, `"ab"`, `"abc"`.

| Trạng thái `k` | Đang chờ ký tự | Gặp ký tự đó thì |
|---:|:---:|---|
| 0 | `a` | sang `k = 1` |
| 1 | `b` | sang `k = 2` |
| 2 | `c` | sang `k = 3` (đã chứa `"abc"`) |
| 3 | — | giữ nguyên (đã hỏng) |

Điều hay: khi **thêm một ký tự** vào cuối chuỗi, ta chỉ cần cập nhật `k` trong `O(1)`, không phải kiểm tra lại từ đầu.

In [2]:
u = "babcc"
k = 0
for i, ch in enumerate(u):
    if k < 3 and ch == "abc"[k]:
        k += 1
    print(f"thêm u[{i}] = {ch!r} -> đã khớp {k} ký tự: {'abc'[:k]!r}")

thêm u[0] = 'b' -> đã khớp 0 ký tự: ''
thêm u[1] = 'a' -> đã khớp 1 ký tự: 'a'
thêm u[2] = 'b' -> đã khớp 2 ký tự: 'ab'
thêm u[3] = 'c' -> đã khớp 3 ký tự: 'abc'
thêm u[4] = 'c' -> đã khớp 3 ký tự: 'abc'


## 2.3. Tính chất đơn điệu

Nếu `s[i..j]` đã chứa `"abc"` như subsequence, thì mọi substring **chứa** `s[i..j]` (dài hơn về bên trái hoặc bên phải) cũng chứa — vì ba vị trí `a, b, c` vẫn còn nguyên trong đó.

Hệ quả, giống bài 4102:

- Cố định điểm đầu `i`: các điểm cuối hợp lệ là một dãy liên tiếp `i, i+1, ..., e - 1` (với `e` là vị trí đầu tiên làm "hỏng").
- Cố định điểm cuối `r`: các điểm đầu hợp lệ là một dãy liên tiếp `L + 1, ..., r`.

Hai góc nhìn này sinh ra hai lời giải `O(n)` ở mục 5–8.

---

# 3. Cách 1 — Brute Force

Thử mọi substring, kiểm tra bằng `is_subsequence`.

In [3]:
def count_valid_bruteforce(s: str) -> int:
    n = len(s)
    count = 0
    for i in range(n):
        for j in range(i, n):
            if not is_subsequence("abc", s[i:j + 1]):
                count += 1
    return count


print(count_valid_bruteforce("abac"))   # 9
print(count_valid_bruteforce("abc"))    # 5
print(count_valid_bruteforce("ac"))     # 3

9
5
3


Có `n(n+1)/2` substring, mỗi lần kiểm tra tốn `O(n)` → **`O(n³)`**. Với `n = 10^5` là không thể.

---

# 4. Cách 2 — O(n²): kéo dài điểm cuối, cập nhật trạng thái

Cố định điểm đầu `i`, kéo `j` sang phải và cập nhật trạng thái `k` (mục 2.2) trong `O(1)`. Khi `k` đạt `3`, nhờ tính đơn điệu, mọi `j` lớn hơn đều hỏng → `break`.

In [4]:
def count_valid_extend(s: str) -> int:
    n = len(s)
    count = 0
    for i in range(n):
        k = 0
        for j in range(i, n):
            if k < 3 and s[j] == "abc"[k]:
                k += 1
            if k == 3:
                break                  # s[i..j] và mọi đoạn dài hơn đều chứa "abc"
            count += 1
    return count


print(count_valid_extend("abac"))   # 9
print(count_valid_extend("abc"))    # 5
print(count_valid_extend("ac"))     # 3

9
5
3


Vẫn **`O(n²)`** ở trường hợp xấu (ví dụ chuỗi toàn `'c'` hoặc `"bbb...b"` không bao giờ khớp đủ). Ta cần trả lời "điểm hỏng đầu tiên" cho mỗi `i` **ngay lập tức**.

---

# 5. Bước đột phá 1: với mỗi điểm đầu `i`, nhảy thẳng tới điểm hỏng

Với điểm đầu `i`, khớp tham lam cho biết:

1. `pa` = vị trí `'a'` **đầu tiên** từ `i` trở đi,
2. `pb` = vị trí `'b'` đầu tiên **sau** `pa`,
3. `pc` = vị trí `'c'` đầu tiên **sau** `pb`.

Khi đó `s[i..j]` chứa `"abc"` **khi và chỉ khi** `j >= pc`. Vậy các điểm cuối hợp lệ là `i, ..., pc - 1`:

```text
số substring hợp lệ bắt đầu tại i  =  pc - i
```

(Nếu không tìm được `pa`, `pb` hoặc `pc`, coi như vị trí đó là `n` — "không bao giờ hỏng" — và công thức cho `n - i`, tức mọi điểm cuối đều hợp lệ.)

Câu hỏi còn lại: tìm "vị trí ký tự `ch` đầu tiên từ chỉ số `x` trở đi" trong `O(1)` như thế nào?

## Bảng vị trí kế tiếp `nxt`

Định nghĩa `nxt[ch][x]` = chỉ số nhỏ nhất `>= x` mà `s[chỉ số] == ch`, hoặc `n` nếu không có.

Xây từ **phải sang trái**: `nxt[ch][x]` bằng `x` nếu `s[x] == ch`, ngược lại bằng `nxt[ch][x + 1]`. Ta thêm một cột `nxt[ch][n] = n` làm lính canh.

In [5]:
def build_next(s: str) -> dict:
    n = len(s)
    nxt = {ch: [n] * (n + 1) for ch in "abc"}     # cột n là lính canh
    for x in range(n - 1, -1, -1):
        for ch in "abc":
            nxt[ch][x] = nxt[ch][x + 1]
        nxt[s[x]][x] = x
    return nxt


s = "abacbc"
nxt = build_next(s)
print("x      :", list(range(len(s) + 1)))
print("s[x]   :", list(s) + ["(n)"])
for ch in "abc":
    print(f"nxt[{ch}] :", nxt[ch])

x      : [0, 1, 2, 3, 4, 5, 6]
s[x]   : ['a', 'b', 'a', 'c', 'b', 'c', '(n)']
nxt[a] : [0, 2, 2, 6, 6, 6, 6]
nxt[b] : [1, 1, 4, 4, 4, 6, 6]
nxt[c] : [3, 3, 3, 3, 5, 5, 6]


Đọc bảng: `nxt['b'][2] = 4` nghĩa là "từ chỉ số 2 trở đi, chữ `b` đầu tiên ở chỉ số 4".

## Tính tay với `s = "abacbc"` (`n = 6`)

| i | pa = nxt[a][i] | pb = nxt[b][pa+1] | pc = nxt[c][pb+1] | pc - i |
|---:|---:|---:|---:|---:|
| 0 | 0 | 1 | 3 | 3 |
| 1 | 2 | 4 | 5 | 4 |
| 2 | 2 | 4 | 5 | 3 |
| 3 | 6 (không có) | — | 6 | 3 |
| 4 | 6 | — | 6 | 2 |
| 5 | 6 | — | 6 | 1 |

Tổng: `3 + 4 + 3 + 3 + 2 + 1 = 16`.

Ví dụ ở `i = 1`: đoạn `s[1..4] = "bacb"` vẫn hợp lệ, đến `s[1..5] = "bacbc"` thì có `a(2) b(4) c(5)` → hỏng. Đúng `5 - 1 = 4` điểm cuối hợp lệ.

# 6. Cách 3 — Bảng vị trí kế tiếp: O(n) thời gian, O(n) bộ nhớ

In [6]:
def count_valid_next(s: str) -> int:
    n = len(s)
    nxt = build_next(s)
    answer = 0
    for i in range(n):
        pa = nxt['a'][i]
        pb = nxt['b'][pa + 1] if pa < n else n      # không có 'a' thì cũng không có điểm hỏng
        pc = nxt['c'][pb + 1] if pb < n else n
        answer += pc - i
    return answer


print(count_valid_next("abac"))     # 9
print(count_valid_next("abc"))      # 5
print(count_valid_next("ac"))       # 3
print(count_valid_next("abacbc"))   # 16

9
5
3
16


In [7]:
def trace_next(s: str) -> int:
    n = len(s)
    nxt = build_next(s)
    answer = 0
    for i in range(n):
        pa = nxt['a'][i]
        pb = nxt['b'][pa + 1] if pa < n else n
        pc = nxt['c'][pb + 1] if pb < n else n
        answer += pc - i
        show = lambda p: str(p) if p < n else "n"
        print(f"i = {i}: pa = {show(pa):>1}, pb = {show(pb):>1}, pc = {show(pc):>1} -> "
              f"điểm cuối hợp lệ {i}..{pc - 1}, cộng {pc - i}, tổng = {answer}")
    return answer


trace_next("abacbc")

i = 0: pa = 0, pb = 1, pc = 3 -> điểm cuối hợp lệ 0..2, cộng 3, tổng = 3
i = 1: pa = 2, pb = 4, pc = 5 -> điểm cuối hợp lệ 1..4, cộng 4, tổng = 7
i = 2: pa = 2, pb = 4, pc = 5 -> điểm cuối hợp lệ 2..4, cộng 3, tổng = 10
i = 3: pa = n, pb = n, pc = n -> điểm cuối hợp lệ 3..5, cộng 3, tổng = 13
i = 4: pa = n, pb = n, pc = n -> điểm cuối hợp lệ 4..5, cộng 2, tổng = 15
i = 5: pa = n, pb = n, pc = n -> điểm cuối hợp lệ 5..5, cộng 1, tổng = 16


16

Cách 3 đã đủ nhanh (`O(n)`), nhưng tốn thêm bảng `3 × (n + 1)` số. Liệu có thể chỉ dùng vài biến?

---

# 7. Bước đột phá 2: cố định điểm cuối, "chuyền vị trí" qua từng chặng

Lần này, với mỗi **điểm cuối `r`**, ta muốn biết:

> `L` = điểm đầu **lớn nhất** sao cho `s[L..r]` vẫn chứa `"abc"` như subsequence (hoặc `-1` nếu không có).

Khi đó các điểm đầu `0..L` đều hỏng, các điểm đầu `L+1..r` đều hợp lệ → cộng `r - L` (y như bài 4102).

Để tìm `L`, ta khớp **ngược**: muốn điểm đầu càng lớn càng tốt thì phải chọn chữ `a` **muộn nhất** có thể. Ta duy trì ba biến:

| Biến | Ý nghĩa (xét đoạn `s[0..r]`) |
|---|---|
| `la` | điểm đầu lớn nhất để có `"a"` — chính là vị trí chữ `a` gần nhất |
| `lab` | điểm đầu lớn nhất để có `"ab"` như subsequence |
| `labc` | điểm đầu lớn nhất để có `"abc"` như subsequence |

Khi đọc ký tự `s[r]`:

- Gặp `'a'`: `la = r` (chữ `a` này là chữ `a` muộn nhất).
- Gặp `'b'`: chữ `b` này có thể ghép với chữ `a` muộn nhất **trước nó** → `lab = la`.
- Gặp `'c'`: chữ `c` này ghép với cặp `"ab"` tốt nhất **trước nó** → `labc = lab`.

Mỗi ký tự **chuyền** vị trí từ chặng trước sang chặng của mình: `a → ab → abc`. Cuối mỗi bước, cộng `r - labc`.

Vì sao khi gặp `'b'` thì `lab = la` luôn tốt hơn giá trị `lab` cũ? Vì `la` hiện tại là chữ `a` muộn nhất trong cả đoạn đã đọc, nên không có cặp `"ab"` nào cho điểm đầu lớn hơn.

## Tính tay với `s = "abacbc"`

| r | s[r] | la | lab | labc | r - labc | tổng |
|---:|:---:|---:|---:|---:|---:|---:|
| 0 | a | **0** | -1 | -1 | 1 | 1 |
| 1 | b | 0 | **0** | -1 | 2 | 3 |
| 2 | a | **2** | 0 | -1 | 3 | 6 |
| 3 | c | 2 | 0 | **0** | 3 | 9 |
| 4 | b | 2 | **2** | 0 | 4 | 13 |
| 5 | c | 2 | 2 | **2** | 3 | 16 |

Cùng ra `16` như cách 3. Ví dụ ở `r = 5`: `labc = 2` nghĩa là `s[2..5] = "acbc"` vẫn chứa `a(2) b(4) c(5)`, còn `s[3..5] = "cbc"` thì không → các điểm đầu `3, 4, 5` hợp lệ.

# 8. Cách 4 — Chuyền vị trí: O(n) thời gian, O(1) bộ nhớ

In [8]:
def count_valid_relay(s: str) -> int:
    la = lab = labc = -1
    answer = 0
    for r, ch in enumerate(s):
        if ch == 'a':
            la = r
        elif ch == 'b':
            lab = la
        else:                      # ch == 'c'
            labc = lab
        answer += r - labc
    return answer


print(count_valid_relay("abac"))     # 9
print(count_valid_relay("abc"))      # 5
print(count_valid_relay("ac"))       # 3
print(count_valid_relay("abacbc"))   # 16

9
5
3
16


In [9]:
def trace_relay(s: str) -> int:
    la = lab = labc = -1
    answer = 0
    print(f"s = {s!r}")
    print(" r | ch |  la | lab | labc | r - labc | tổng")
    for r, ch in enumerate(s):
        if ch == 'a':
            la = r
        elif ch == 'b':
            lab = la
        else:
            labc = lab
        answer += r - labc
        print(f"{r:>2} |  {ch} | {la:>3} | {lab:>3} | {labc:>4} | {r - labc:>8} | {answer:>4}")
    return answer


trace_relay("abacbc")

s = 'abacbc'
 r | ch |  la | lab | labc | r - labc | tổng
 0 |  a |   0 |  -1 |   -1 |        1 |    1
 1 |  b |   0 |   0 |   -1 |        2 |    3
 2 |  a |   2 |   0 |   -1 |        3 |    6
 3 |  c |   2 |   0 |    0 |        3 |    9
 4 |  b |   2 |   2 |    0 |        4 |   13
 5 |  c |   2 |   2 |    2 |        3 |   16


16

# 9. Phiên bản đúng format LeetCode

Ta nộp cách 4: ngắn nhất, nhanh nhất, ít bộ nhớ nhất.

In [10]:
class Solution:
    def countValidSubstrings(self, s: str) -> int:
        la = lab = labc = -1
        answer = 0
        for r, ch in enumerate(s):
            if ch == 'a':
                la = r
            elif ch == 'b':
                lab = la
            else:
                labc = lab
            answer += r - labc
        return answer

In [11]:
sol = Solution()

tests = [
    ("abac", 9, "ví dụ 1"),
    ("abc", 5, "ví dụ 2"),
    ("ac", 3, "ví dụ 3"),
    ("a", 1, "một ký tự"),
    ("cba", 6, "ngược thứ tự: luôn hợp lệ"),
    ("aaabbbccc", 36, "khối a, b, c"),
    ("abacbc", 16, "ví dụ tự tính"),
    ("bbbb", 10, "không có a"),
]

for s, expected, name in tests:
    result = sol.countValidSubstrings(s)
    assert result == expected, (name, s, result, expected)
    print(f"PASS | {name:26} | s = {s!r:12} -> {result}")

PASS | ví dụ 1                    | s = 'abac'       -> 9
PASS | ví dụ 2                    | s = 'abc'        -> 5
PASS | ví dụ 3                    | s = 'ac'         -> 3
PASS | một ký tự                  | s = 'a'          -> 1
PASS | ngược thứ tự: luôn hợp lệ  | s = 'cba'        -> 6
PASS | khối a, b, c               | s = 'aaabbbccc'  -> 36
PASS | ví dụ tự tính              | s = 'abacbc'     -> 16
PASS | không có a                 | s = 'bbbb'       -> 10


In [12]:
import random
import time

random.seed(4104)
big = "".join(random.choice("abc") for _ in range(10**5))
start = time.perf_counter()
result = sol.countValidSubstrings(big)
print(f"n = {len(big):,} -> {result:,} ({time.perf_counter() - start:.3f} giây)")

n = 100,000 -> 803,617 (0.006 giây)


Kiểm tra lại test `"aaabbbccc" -> 36`: tổng có `45` substring. Substring hỏng phải chứa ít nhất một `a`, một `b`, một `c` → bắt đầu trong khối `a` (3 cách) và kết thúc trong khối `c` (3 cách) → `9` substring hỏng. `45 - 9 = 36`. ✔

---

# 10. Vì sao cách 4 đúng?

**Bất biến:** sau khi đọc `s[0..r]`,

- `la` = chỉ số lớn nhất `<= r` có `s[la] = 'a'` (hoặc `-1`);
- `lab` = điểm đầu `L` lớn nhất sao cho `s[L..r]` chứa `"ab"` (hoặc `-1`);
- `labc` = điểm đầu `L` lớn nhất sao cho `s[L..r]` chứa `"abc"` (hoặc `-1`).

Chứng minh cho `lab` (hai biến kia tương tự):

- `s[L..r]` chứa `"ab"` ⇔ có chữ `b` ở vị trí `q <= r` và chữ `a` ở vị trí `p` với `L <= p < q`.
- Với một chữ `b` cố định tại `q`, điểm đầu tốt nhất là chữ `a` **muộn nhất trước `q`**.
- Chữ `b` càng muộn thì "chữ `a` muộn nhất trước nó" càng muộn (hoặc bằng). Nên chỉ cần quan tâm chữ `b` **gần nhất** → khi gặp `b` tại `r`, `lab = la` (giá trị `la` lúc đó chính là chữ `a` muộn nhất trước `r`).
- Nếu `s[r]` không phải `b`, các cặp `"ab"` không đổi → `lab` giữ nguyên.

Với `labc` lập luận giống hệt, thay "chữ `a`" bằng "cặp `ab`".

Cuối cùng, nhờ tính đơn điệu: `s[i..r]` hỏng ⇔ `i <= labc`. Số điểm đầu hợp lệ là `r - labc`. Cộng qua mọi `r`, mỗi substring được đếm đúng một lần theo điểm cuối của nó.

---

## Trực giác

> Đọc từ trái sang phải, mỗi chữ cái "nhận gậy tiếp sức" từ chữ cái đứng trước nó trong mẫu `abc`, và luôn chọn người chuyền **muộn nhất** để đoạn hỏng càng ngắn càng tốt.

# 11. Độ phức tạp

Gọi `n = len(s)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Brute Force | mọi substring, kiểm tra subsequence | `O(n³)` | `O(n)` |
| 2. Kéo dài + trạng thái | cố định `i`, cập nhật `k` trong `O(1)` | `O(n²)` | `O(1)` |
| 3. Bảng vị trí kế tiếp | cố định điểm đầu, nhảy tới `pc` | `O(n)` | `O(n)` |
| 4. Chuyền vị trí | cố định điểm cuối, 3 biến | `O(n)` | `O(1)` |

Kết quả có thể tới khoảng `5·10^9` → trong C++/Java dùng kiểu 64-bit.

# 12. Đối chiếu các cách trên dữ liệu ngẫu nhiên

In [13]:
random.seed(2026)
for _ in range(2000):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 14)))
    expected = count_valid_bruteforce(s)
    assert count_valid_extend(s) == expected, s
    assert count_valid_next(s) == expected, s
    assert count_valid_relay(s) == expected, s
    assert sol.countValidSubstrings(s) == expected, s

print("2000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!")

2000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!


# 13. Những lỗi dễ mắc

## Lỗi 1 — Giải nhầm thành bài substring (4102)

Nếu chỉ kiểm tra `"abc"` **liền nhau**, `"abac"` sẽ ra `10` thay vì `9`.

In [14]:
def wrong_as_substring(s: str) -> int:
    last = -1
    answer = 0
    for r in range(len(s)):
        if r >= 2 and s[r - 2:r + 1] == "abc":
            last = r - 2
        answer += r - last
    return answer


print("Sai :", wrong_as_substring("abac"))
print("Đúng:", sol.countValidSubstrings("abac"))

Sai : 10
Đúng: 9


## Lỗi 2 — Khi gặp `'b'` lại lấy vị trí của chính nó

Code sai:

```python
elif ch == 'b':
    lab = r          # SAI: lab phải là vị trí của chữ 'a' đứng trước, không phải của 'b'
```

`lab` là **điểm đầu** của cặp `"ab"`, tức là vị trí chữ `a`. Lấy `r` làm điểm đầu sẽ biến mọi đoạn bắt đầu tại chữ `b` thành "hỏng" dù không có chữ `a` nào.

In [15]:
def wrong_lab_is_r(s: str) -> int:
    la = lab = labc = -1
    answer = 0
    for r, ch in enumerate(s):
        if ch == 'a':
            la = r
        elif ch == 'b':
            lab = r                    # SAI
        else:
            labc = lab
        answer += r - labc
    return answer


print("Sai :", wrong_lab_is_r("bc"))       # "bc" không có 'a' nên phải có 3 substring hợp lệ
print("Đúng:", sol.countValidSubstrings("bc"))

Sai : 2
Đúng: 3


## Lỗi 3 — Quên trường hợp "không tìm thấy" trong bảng vị trí kế tiếp

Ở cách 3, nếu không có chữ `a` nào từ `i` trở đi thì `pa = n`. Viết vội `nxt['b'][pa + 1]` sẽ truy cập chỉ số `n + 1` — vượt khỏi bảng (bảng chỉ có các cột `0..n`).

In [16]:
def wrong_no_guard(s: str) -> int:
    n = len(s)
    nxt = build_next(s)
    answer = 0
    for i in range(n):
        pa = nxt['a'][i]
        pb = nxt['b'][pa + 1]          # SAI: pa có thể bằng n
        pc = nxt['c'][pb + 1]
        answer += pc - i
    return answer


try:
    wrong_no_guard("ab")
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


## Lỗi 4 — Khớp tham lam sai chiều

- Khi cố định **điểm đầu** (cách 3), phải khớp **sớm nhất** để biết điểm hỏng đến sớm nhất là khi nào.
- Khi cố định **điểm cuối** (cách 4), phải khớp **muộn nhất** để biết điểm đầu hỏng lớn nhất.

Trộn lẫn hai chiều (ví dụ cách 4 mà dùng chữ `a` **đầu tiên** thay vì gần nhất) sẽ đếm thiếu hoặc thừa.

---

## Lỗi 5 — Cộng `r - labc + 1`

Các điểm đầu hợp lệ là `labc + 1, ..., r`: có `r - labc` phần tử. Thử với `s = "a"`: `labc = -1`, `r = 0` → phải ra `1`.

# 14. Mẹo nhận diện

1. Thấy chữ **subsequence** → nghĩ ngay tới **khớp tham lam**: từ trái sang thì khớp **sớm nhất**, từ phải sang (hoặc theo điểm cuối) thì khớp **muộn nhất**.
2. Đếm **substring thỏa điều kiện có tính đơn điệu** → cố định một đầu, tìm ranh giới của đầu kia (giống 4102).
3. Cần hỏi nhiều lần "ký tự `ch` đầu tiên từ vị trí `x` trở đi" → **bảng vị trí kế tiếp** `nxt[ch][x]`, xây từ phải sang trái.
4. Mẫu là một chuỗi ngắn cố định (`"abc"`) → có thể duy trì **một biến cho mỗi tiền tố của mẫu** (`a`, `ab`, `abc`) và chuyền giá trị giữa chúng.

Các bài cùng họ: *Is Subsequence* (LeetCode 392), *Number of Matching Subsequences* (792), *Count Subarrays With Fixed Bounds* (2444), *Number of Substrings Containing All Three Characters* (1358).

# 15. Biến thể

## Biến thể 1 — Đếm substring **có chứa** `"abc"` như subsequence

Đếm phần bù: với mỗi `r`, số điểm đầu hỏng là `labc + 1`.

In [17]:
def count_containing_abc(s: str) -> int:
    la = lab = labc = -1
    bad = 0
    for r, ch in enumerate(s):
        if ch == 'a':
            la = r
        elif ch == 'b':
            lab = la
        else:
            labc = lab
        bad += labc + 1
    return bad


print(count_containing_abc("abac"))        # 1
print(count_containing_abc("aaabbbccc"))   # 9
n = len("abacbc")
print(count_containing_abc("abacbc") + sol.countValidSubstrings("abacbc") == n * (n + 1) // 2)

1
9
True


## Biến thể 2 — Nhiều câu hỏi "từ có phải subsequence của `s` không?"

Bảng vị trí kế tiếp cho phép kiểm tra mỗi từ `w` trong `O(len(w))` thay vì `O(len(s))` — ý tưởng của bài *Number of Matching Subsequences* (792).

In [18]:
def build_next_general(s: str) -> list:
    n = len(s)
    nxt = [None] * (n + 1)
    nxt[n] = {}
    for x in range(n - 1, -1, -1):
        nxt[x] = dict(nxt[x + 1])
        nxt[x][s[x]] = x
    return nxt


def is_subseq_fast(word: str, nxt: list, n: int) -> bool:
    pos = 0
    for ch in word:
        if ch not in nxt[pos]:
            return False
        pos = nxt[pos][ch] + 1        # nhảy qua ký tự vừa khớp
    return True


s = "abcde"
nxt_general = build_next_general(s)
for w in ["a", "bb", "acd", "ace", "ea"]:
    print(f"{w!r:6} là subsequence của {s!r}? {is_subseq_fast(w, nxt_general, len(s))}")

'a'    là subsequence của 'abcde'? True
'bb'   là subsequence của 'abcde'? False
'acd'  là subsequence của 'abcde'? True
'ace'  là subsequence của 'abcde'? True
'ea'   là subsequence của 'abcde'? False


(Bảng ở biến thể 2 dùng `dict` cho gọn khi bảng chữ cái lớn; với 3 chữ cái như bài gốc, ba list như `build_next` tiết kiệm hơn.)

---

# 16. Template tư duy

```python
# Đếm substring KHÔNG chứa mẫu cố định P = p1 p2 ... pm như subsequence
best = [-1] * (m + 1)            # best[k]: điểm đầu lớn nhất để có p1..pk
answer = 0
for r, ch in enumerate(s):
    for k in range(m, 0, -1):    # đi từ chặng cao xuống để dùng giá trị CŨ
        if P[k - 1] == ch:
            best[k] = r if k == 1 else best[k - 1]
    answer += r - best[m]
```

Với `P = "abc"`, `best[1], best[2], best[3]` chính là `la, lab, labc`.

# 17. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng `r | ch | la | lab | labc | r - labc` cho:

```python
s = "babcac"
```

Kết quả là bao nhiêu? Kiểm tra bằng cell dưới.

In [19]:
# Kiểm tra Bài 1 sau khi đã tính tay
trace_relay("babcac");

s = 'babcac'
 r | ch |  la | lab | labc | r - labc | tổng
 0 |  b |  -1 |  -1 |   -1 |        1 |    1
 1 |  a |   1 |  -1 |   -1 |        2 |    3
 2 |  b |   1 |   1 |   -1 |        3 |    6
 3 |  c |   1 |   1 |    1 |        2 |    8
 4 |  a |   4 |   1 |    1 |        3 |   11
 5 |  c |   4 |   1 |    1 |        4 |   15


## Bài 2 — Dự đoán output

```python
print(is_subsequence("abc", "cabbac"))
print(is_subsequence("abc", "aabbcc"))
print(is_subsequence("", "xyz"))
print(is_subsequence("ac", "ca"))
```

In [20]:
print(is_subsequence("abc", "cabbac"))
print(is_subsequence("abc", "aabbcc"))
print(is_subsequence("", "xyz"))
print(is_subsequence("ac", "ca"))

True
True
True
False


## Bài 3 — Hoàn thiện cách chuyền vị trí

In [21]:
def exercise_3_relay(s: str) -> int:
    la = lab = labc = -1
    answer = 0
    for r, ch in enumerate(s):
        # TODO 1: cập nhật la / lab / labc tùy theo ch
        # TODO 2: cộng số substring hợp lệ kết thúc tại r
        pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_relay("abac") == 9
# assert exercise_3_relay("abacbc") == 16

## Bài 4 — Tự xây bảng vị trí kế tiếp

Viết hàm `build_next_exercise(s)` trả về dict `{ch: list}` như `build_next`, với lính canh ở cột `n`.

```python
build_next_exercise("aba")['b']   -> [1, 1, 3, 3]
```

In [22]:
def exercise_4_build_next(s: str) -> dict:
    # TODO
    raise NotImplementedError


# assert exercise_4_build_next("aba")['b'] == [1, 1, 3, 3]
# assert exercise_4_build_next("aba")['a'] == [0, 2, 2, 3]

## Bài 5 — Mẫu ngắn hơn: cấm `"ab"`

Đếm số substring **không** chứa `"ab"` như subsequence (chuỗi chỉ gồm `a, b, c`).

```python
count_without_ab("ab")    -> 2
count_without_ab("ba")    -> 3
count_without_ab("abab")  -> 5
```

In [23]:
def exercise_5_count_without_ab(s: str) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_count_without_ab("ab") == 2
# assert exercise_5_count_without_ab("ba") == 3
# assert exercise_5_count_without_ab("abab") == 5

## Bài 6 — Mini challenge: mẫu `P` bất kỳ

Đếm số substring của `s` **không** chứa `P` như subsequence, với `P` là chuỗi bất kỳ (có thể có ký tự lặp, ví dụ `"aba"`).

```python
count_without_pattern("abac", "abc")   -> 9
count_without_pattern("abab", "aba")   -> 8
count_without_pattern("aaaa", "aa")    -> 4
```

Gợi ý: dùng template ở mục 16. Vì sao phải duyệt `k` từ **cao xuống thấp**?

In [24]:
def exercise_6_count_without_pattern(s: str, P: str) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_count_without_pattern("abac", "abc") == 9
# assert exercise_6_count_without_pattern("abab", "aba") == 8
# assert exercise_6_count_without_pattern("aaaa", "aa") == 4

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: chữ `a` cập nhật `la`, chữ `b` cập nhật `lab`, chữ `c` cập nhật `labc`.
- Mức 2: `la = r`, `lab = la`, `labc = lab`.
- Mức 3: `answer += r - labc`.

**Bài 4**

- Mức 1: khởi tạo mọi cột bằng `n`, rồi đi từ phải sang trái.
- Mức 2: mỗi cột `x` sao chép từ cột `x + 1`.
- Mức 3: sau đó ghi đè `nxt[s[x]][x] = x`.

**Bài 5**

- Mức 1: chỉ cần hai chặng: `la` và `lab`.
- Mức 2: chữ `c` không ảnh hưởng gì.
- Mức 3: `answer += r - lab`.

**Bài 6**

- Mức 1: `best[k]` là điểm đầu lớn nhất để có `P[:k]`.
- Mức 2: gặp ký tự `ch == P[k-1]` thì `best[k] = best[k-1]` (với `best[0]` coi như `r`).
- Mức 3: duyệt `k` giảm dần để `best[k-1]` vẫn là giá trị **trước khi** đọc `s[r]` — nếu duyệt tăng dần, với `P = "aa"` một chữ `a` sẽ bị dùng cho cả hai chặng.

# 18. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

`s = "babcac"`:

| r | ch | la | lab | labc | r - labc | tổng |
|---:|:---:|---:|---:|---:|---:|---:|
| 0 | b | -1 | -1 | -1 | 1 | 1 |
| 1 | a | 1 | -1 | -1 | 2 | 3 |
| 2 | b | 1 | 1 | -1 | 3 | 6 |
| 3 | c | 1 | 1 | 1 | 2 | 8 |
| 4 | a | 4 | 1 | 1 | 3 | 11 |
| 5 | c | 4 | 1 | 1 | 4 | 15 |

Kết quả: `15`. Để ý ở `r = 0`: chữ `b` đầu tiên chưa có chữ `a` nào trước nó nên `lab = la = -1`.

## Đáp án Bài 2

```text
True     (a ở 1, b ở 2, c ở 5)
True
True     (chuỗi rỗng là subsequence của mọi chuỗi)
False
```

In [25]:
# Đáp án Bài 3
def answer_3_relay(s: str) -> int:
    la = lab = labc = -1
    answer = 0
    for r, ch in enumerate(s):
        if ch == 'a':
            la = r
        elif ch == 'b':
            lab = la
        else:
            labc = lab
        answer += r - labc
    return answer


assert answer_3_relay("abac") == 9
assert answer_3_relay("abacbc") == 16
assert answer_3_relay("babcac") == 15
print("Bài 3: PASS")

Bài 3: PASS


In [26]:
# Đáp án Bài 4
def answer_4_build_next(s: str) -> dict:
    n = len(s)
    nxt = {ch: [n] * (n + 1) for ch in "abc"}
    for x in range(n - 1, -1, -1):
        for ch in "abc":
            nxt[ch][x] = nxt[ch][x + 1]
        nxt[s[x]][x] = x
    return nxt


assert answer_4_build_next("aba")['b'] == [1, 1, 3, 3]
assert answer_4_build_next("aba")['a'] == [0, 2, 2, 3]
print("Bài 4: PASS")

Bài 4: PASS


In [27]:
# Đáp án Bài 5
def answer_5_count_without_ab(s: str) -> int:
    la = lab = -1
    answer = 0
    for r, ch in enumerate(s):
        if ch == 'a':
            la = r
        elif ch == 'b':
            lab = la
        answer += r - lab
    return answer


assert answer_5_count_without_ab("ab") == 2
assert answer_5_count_without_ab("ba") == 3
assert answer_5_count_without_ab("abab") == 5
print("Bài 5: PASS")

Bài 5: PASS


In [28]:
# Đáp án Bài 6
def answer_6_count_without_pattern(s: str, P: str) -> int:
    m = len(P)
    best = [-1] * (m + 1)
    answer = 0
    for r, ch in enumerate(s):
        for k in range(m, 0, -1):              # từ cao xuống thấp
            if P[k - 1] == ch:
                best[k] = r if k == 1 else best[k - 1]
        answer += r - best[m]
    return answer


def brute_without_pattern(s: str, P: str) -> int:
    n = len(s)
    return sum(1 for i in range(n) for j in range(i, n) if not is_subsequence(P, s[i:j + 1]))


assert answer_6_count_without_pattern("abac", "abc") == 9
assert answer_6_count_without_pattern("abab", "aba") == 8
assert answer_6_count_without_pattern("aaaa", "aa") == 4
random.seed(6)
for _ in range(500):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 10)))
    P = "".join(random.choice("ab") for _ in range(random.randint(1, 3)))
    assert answer_6_count_without_pattern(s, P) == brute_without_pattern(s, P), (s, P)
print("Bài 6: PASS (kể cả 500 test ngẫu nhiên với mẫu có ký tự lặp)")

Bài 6: PASS (kể cả 500 test ngẫu nhiên với mẫu có ký tự lặp)


# 19. Tổng kết

Bài **Count Valid Substrings II** dạy cách xử lý **subsequence** trong bài đếm substring:

1. **Khớp tham lam**: từ trái sang chọn sớm nhất, theo điểm cuối chọn muộn nhất.
2. **Tính đơn điệu** → mỗi điểm đầu (hoặc điểm cuối) có một ranh giới duy nhất.
3. Tìm ranh giới nhanh bằng **bảng vị trí kế tiếp** (`O(n)` bộ nhớ) hoặc **chuyền vị trí** qua các chặng `a → ab → abc` (`O(1)` bộ nhớ).

## Công thức nhớ nhanh

```text
gặp 'a' -> la = r
gặp 'b' -> lab = la
gặp 'c' -> labc = lab
mỗi r   -> answer += r - labc
```

## Phiên bản nên nhớ

```python
class Solution:
    def countValidSubstrings(self, s: str) -> int:
        la = lab = labc = -1
        answer = 0
        for r, ch in enumerate(s):
            if ch == 'a':
                la = r
            elif ch == 'b':
                lab = la
            else:
                labc = lab
            answer += r - labc
        return answer
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(1)
```

# 20. Checklist trước khi submit

- [ ] Đề cấm `"abc"` như **subsequence** (không cần liền nhau)? Ví dụ `"abac"` phải ra `9`.
- [ ] Khi gặp `'b'`: `lab = la` (vị trí chữ `a`), **không** phải `lab = r`?
- [ ] Khi gặp `'c'`: `labc = lab`?
- [ ] Cộng `r - labc` (không `+ 1`)? Kiểm tra `s = "a"` → `1`.
- [ ] Khởi tạo cả ba biến bằng `-1`?
- [ ] Nếu dùng bảng vị trí kế tiếp: đã chặn trường hợp `pa == n` / `pb == n` để không vượt chỉ số?
- [ ] Đã test chuỗi không có `a`, chuỗi ngược thứ tự `"cba"`, chuỗi khối `"aaabbbccc"`?